# Quantitative Asset Forecasting — Gold & Stock/ETF Portfolio Modeling

A robust **quant-style research notebook** for forecasting and backtesting across:

- **GLD** — Gold ETF
- **SPY** — S&P 500 ETF
- **QQQ** — Nasdaq ETF
- **AAPL** — Apple stock

## Models compared
- Naive baseline
- ARIMA
- Attention-based PyTorch LSTM

## Evaluation
- RMSE, MAE, MAPE
- Directional accuracy
- Strategy cumulative return
- Buy-and-hold return
- Sharpe ratio
- Maximum drawdown

## Upgrades in this version
- 10 years of data
- market data caching
- RSI, MACD, Bollinger Bands
- faster ARIMA walk-forward
- PyTorch LSTM with attention
- scheduler + early stopping
- optional direction target
- safer parallel execution helper

## Setup

Run the next cell once if you need to install the required packages.

In [16]:
# Uncomment if needed
# !pip install yfinance pandas numpy matplotlib scikit-learn statsmodels torch

## Imports and Configuration

In [17]:
import warnings
warnings.filterwarnings("ignore")

import os
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from concurrent.futures import ProcessPoolExecutor, as_completed
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

warnings.simplefilter("ignore", ConvergenceWarning)

np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

TICKERS = ["GLD", "SPY", "QQQ", "AAPL"]
EXTRA_TICKERS = ["TLT", "UUP"]
ALL_TICKERS = sorted(set(TICKERS + EXTRA_TICKERS))
START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

CACHE_FILE = "market_data_cache.pkl"

TRAIN_RATIO = 0.75
SEQUENCE_LENGTH = 30
HORIZON = 5
WALK_FORWARD_SPLITS = 3
MIN_TRAIN_SIZE = 900
MIN_TEST_SIZE = 180

EPOCHS = 60
BATCH_SIZE = 64
LEARNING_RATE = 1e-3
EARLY_STOP_PATIENCE = 10

ARIMA_ORDER = (1, 1, 1)
ARIMA_REFIT_EVERY = 5

INITIAL_CAPITAL = 10000
CONFIDENCE_THRESHOLD = 0.60
CONSENSUS_GAP = 0.05

# Choose one:
# TARGET_MODE = "price"
TARGET_MODE = "direction"

Using device: cpu


## Data Retrieval with Caching

Historical market data is downloaded using `yfinance`. To make repeated experiments faster, the notebook caches the downloaded dataset locally.

In [18]:
def download_market_data(tickers, start, end):
    data = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False
    )
    if data.empty:
        raise ValueError("No data downloaded.")
    return data


def has_required_tickers(data, required_tickers):
    try:
        available = set(data["Close"].columns)
    except Exception:
        return False
    return set(required_tickers).issubset(available)


if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "rb") as f:
        market_data = pickle.load(f)
    if has_required_tickers(market_data, ALL_TICKERS):
        print("Loaded market data from cache.")
    else:
        print("Cache missing required tickers; refreshing download.")
        market_data = download_market_data(ALL_TICKERS, START_DATE, END_DATE)
        with open(CACHE_FILE, "wb") as f:
            pickle.dump(market_data, f)
else:
    market_data = download_market_data(ALL_TICKERS, START_DATE, END_DATE)
    with open(CACHE_FILE, "wb") as f:
        pickle.dump(market_data, f)
    print("Downloaded and cached market data.")

print("Market data shape:", market_data.shape)
market_data.head()

Loaded market data from cache.
Market data shape: (2765, 30)


Price           Close                                                \
Ticker           AAPL         GLD        QQQ         SPY        TLT   
Date                                                                  
2015-01-02  24.214895  114.080002  94.784431  170.589630  94.446381   
2015-01-05  23.532719  115.800003  93.394066  167.508865  95.930023   
2015-01-06  23.534933  117.120003  92.141815  165.931091  97.658386   
2015-01-07  23.864939  116.430000  93.329620  167.998779  97.465485   
2015-01-08  24.781891  115.940002  95.115906  170.979919  96.174774   

Price                       High                                     ...  \
Ticker            UUP       AAPL         GLD        QQQ         SPY  ...   
Date                                                                 ...   
2015-01-02  20.216696  24.682228  114.800003  95.944601  171.793724  ...   
2015-01-05  20.258465  24.064282  116.000000  94.480579  169.709428  ...   
2015-01-06  20.300236  23.794069  117.500000  93.688715  168.339254  ...   
2015-01-07  20.375420  23.964606  116.879997  93.550604  168.339247  ...   
2015-01-08  20.458961  24.839477  116.870003  95.300058  171.195832  ...   

Price            Open                                       Volume            \
Ticker            QQQ         SPY        TLT        UUP       AAPL       GLD   
Date                                                                           
2015-01-02  95.539465  171.378523  93.682324  20.149864  212818400   7109600   
2015-01-05  94.370084  169.543350  94.958251  20.308588  257142000   8177400   
2015-01-06  93.532185  167.816097  96.953675  20.300236  263188400  11238300   
2015-01-07  92.749535  167.259721  96.953639  20.408836  160423600   6434200   
2015-01-08  94.121469  169.410459  96.701459  20.458961  237458000   7033700   

Price                                               
Ticker           QQQ        SPY       TLT      UUP  
Date                                                
2015-01-02  31314600  121465900   9432000  1887600  
2015-01-05  36521300  169632600   9789500  2855500  
2015-01-06  66205500  209151400  18331300  2262700  
2015-01-07  37577400  125346700   9762900  2384600  
2015-01-08  40212600  147217800   8055300  1685500  

[5 rows x 30 columns]

## Feature Engineering

This notebook generates a richer feature set than a basic forecasting notebook.

### Core features
- lagged prices
- lagged returns
- moving averages
- momentum
- volatility
- price spreads
- volume changes

### Technical indicators
- RSI
- MACD
- Bollinger Bands

The target can be configured as either:
- next-day **price**
- next-day **direction**

In [19]:
def build_features_for_ticker(market_df, ticker, target_mode="direction"):
    df = pd.DataFrame({
        "Open": market_df["Open"][ticker],
        "High": market_df["High"][ticker],
        "Low": market_df["Low"][ticker],
        "Close": market_df["Close"][ticker],
        "Volume": market_df["Volume"][ticker],
    }).dropna().copy()

    df["next_close"] = df["Close"].shift(-HORIZON)
    df["return_1d"] = df["Close"].pct_change()
    df["return_5d"] = df["Close"].pct_change(5)
    df["return_10d"] = df["Close"].pct_change(10)
    df["return_20d"] = df["Close"].pct_change(20)
    df["intraday_return"] = (df["Close"] - df["Open"]) / df["Open"]

    for lag in [1, 2, 3, 5, 10]:
        df[f"close_lag_{lag}"] = df["Close"].shift(lag)
        df[f"return_lag_{lag}"] = df["return_1d"].shift(lag)

    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_10"] = df["Close"].rolling(10).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df["ma_30"] = df["Close"].rolling(30).mean()
    df["ma_cross"] = (df["ma_5"] / df["ma_20"]) - 1

    df["momentum_5"] = df["Close"] - df["Close"].shift(5)
    df["momentum_10"] = df["Close"] - df["Close"].shift(10)

    df["volatility_5"] = df["return_1d"].rolling(5).std()
    df["volatility_10"] = df["return_1d"].rolling(10).std()
    df["volatility_20"] = df["return_1d"].rolling(20).std()

    df["hl_range"] = df["High"] - df["Low"]
    df["oc_range"] = df["Open"] - df["Close"]

    df["volume_change"] = df["Volume"].pct_change()
    df["volume_ma_5"] = df["Volume"].rolling(5).mean()
    df["volume_ma_20"] = df["Volume"].rolling(20).mean()

    df["price_to_ma20"] = df["Close"] / df["ma_20"]
    df["vol_ratio"] = df["volatility_5"] / df["volatility_20"]
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / loss.replace(0, np.nan)
    df["rsi_14"] = 100 - (100 / (1 + rs))
    df["rsi_centered"] = (df["rsi_14"] - 50) / 50

    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["macd"] = ema12 - ema26
    df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()
    df["macd_hist"] = df["macd"] - df["macd_signal"]

    rolling_mean = df["Close"].rolling(20).mean()
    rolling_std = df["Close"].rolling(20).std()
    df["bb_upper"] = rolling_mean + 2 * rolling_std
    df["bb_lower"] = rolling_mean - 2 * rolling_std
    df["bb_width"] = df["bb_upper"] - df["bb_lower"]

    # TLT (treasuries) and UUP (dollar) are relevant cross-asset signals
    # for GLD (gold), SPY (S&P500), and QQQ (Nasdaq) — all rate-sensitive.
    if ticker in ("GLD", "SPY", "QQQ"):
        for macro_ticker in EXTRA_TICKERS:
            macro_close = market_df["Close"][macro_ticker].reindex(df.index)
            df[f"{macro_ticker.lower()}_return_1d"] = macro_close.pct_change()
            df[f"{macro_ticker.lower()}_return_5d"] = macro_close.pct_change(5)

    if isinstance(df.index, pd.DatetimeIndex):
        df["day_of_week"] = df.index.dayofweek
        df["month"] = df.index.month
    else:
        df["day_of_week"] = 0
        df["month"] = 0

    if target_mode == "price":
        df["target"] = df["next_close"]
    elif target_mode == "direction":
        df["target"] = (df["next_close"] > df["Close"]).astype(int)
    else:
        raise ValueError("target_mode must be 'price' or 'direction'")

    df = df.iloc[:-HORIZON]
    df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

    if len(df) < 200:
        raise ValueError(f"Not enough rows after feature engineering for {ticker}: {len(df)}")

    return df


example_df = build_features_for_ticker(market_data, "GLD", target_mode=TARGET_MODE)
print("Feature dataset shape:", example_df.shape)
example_df.head()

Feature dataset shape: (2731, 53)


,Open,High,Low,Close,Volume,next_close,return_1d,return_5d,return_10d,return_20d,...,bb_upper,bb_lower,bb_width,tlt_return_1d,tlt_return_5d,uup_return_1d,uup_return_5d,day_of_week,month,target
0,118.050003,118.580002,117.790001,117.980003,4108900,115.430000,0.005454,-0.005563,-0.044309,-0.024475,...,126.758753,116.335247,10.423506,-0.010269,-0.021152,-0.000401,-0.005986,4,2,0
1,116.400002,116.540001,115.580002,116.010002,7225200,115.260002,-0.016698,-0.026517,-0.052361,-0.053134,...,126.963568,115.479433,11.484135,-0.015055,-0.034340,-0.001204,-0.006390,1,2,0
2,115.989998,116.529999,114.989998,116.339996,8336000,115.699997,0.002845,-0.017979,-0.038910,-0.063285,...,126.784170,114.872830,11.911340,0.005940,-0.020816,0.000402,-0.007180,2,2,0
3,116.400002,116.540001,115.739998,115.940002,6528700,116.070000,-0.003438,-0.009652,-0.046389,-0.066731,...,126.524793,114.303207,12.221586,-0.006613,-0.029088,0.003616,-0.007154,3,2,1
4,116.099998,116.459999,115.050003,115.279999,6681700,116.160004,-0.005693,-0.017556,-0.053453,-0.079454,...,125.996449,113.836551,12.159899,0.002933,-0.023008,-0.000801,0.001605,4,2,1


## Metrics and Backtesting Helpers

Forecasting metrics:
- RMSE
- MAE
- MAPE
- directional accuracy

Backtesting metrics:
- cumulative return
- buy-and-hold return
- Sharpe ratio
- maximum drawdown

In [20]:
def mape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None))) * 100

def directional_accuracy_from_prev(prev_close, actual_next, pred_next):
    actual_dir = np.sign(actual_next - prev_close)
    pred_dir = np.sign(pred_next - prev_close)
    return (actual_dir == pred_dir).mean() * 100

def sharpe_ratio(returns, periods_per_year=252):
    returns = np.asarray(returns)
    if returns.std() == 0:
        return 0.0
    return (returns.mean() / returns.std()) * np.sqrt(periods_per_year)

def max_drawdown(portfolio_values):
    pv = np.asarray(portfolio_values)
    peaks = np.maximum.accumulate(pv)
    dd = (pv - peaks) / peaks
    return dd.min()

def calculate_strategy_metrics(prev_close, actual_next, pred_next, initial_capital=10000):
    predicted_return = (pred_next - prev_close) / prev_close
    signal = (predicted_return > 0.002).astype(int)
    actual_return = (actual_next - prev_close) / prev_close
    strategy_return = signal * actual_return
    buy_hold_return = actual_return.copy()

    strategy_portfolio = initial_capital * np.cumprod(1 + strategy_return)
    buy_hold_portfolio = initial_capital * np.cumprod(1 + buy_hold_return)

    return {
        "strategy_returns": strategy_return,
        "buy_hold_returns": buy_hold_return,
        "strategy_portfolio": strategy_portfolio,
        "buy_hold_portfolio": buy_hold_portfolio,
        "strategy_cum_return_pct": (strategy_portfolio[-1] / initial_capital - 1) * 100,
        "buy_hold_cum_return_pct": (buy_hold_portfolio[-1] / initial_capital - 1) * 100,
        "sharpe_ratio": sharpe_ratio(strategy_return),
        "max_drawdown": max_drawdown(strategy_portfolio),
    }

def calculate_strategy_metrics_from_direction(prev_close, actual_next, pred_direction, initial_capital=10000):
    signal = np.asarray(pred_direction).astype(int)
    actual_return = (np.asarray(actual_next) - np.asarray(prev_close)) / np.asarray(prev_close)
    strategy_return = signal * actual_return
    buy_hold_return = actual_return.copy()

    strategy_portfolio = initial_capital * np.cumprod(1 + strategy_return)
    buy_hold_portfolio = initial_capital * np.cumprod(1 + buy_hold_return)

    return {
        "strategy_returns": strategy_return,
        "buy_hold_returns": buy_hold_return,
        "strategy_portfolio": strategy_portfolio,
        "buy_hold_portfolio": buy_hold_portfolio,
        "strategy_cum_return_pct": (strategy_portfolio[-1] / initial_capital - 1) * 100,
        "buy_hold_cum_return_pct": (buy_hold_portfolio[-1] / initial_capital - 1) * 100,
        "sharpe_ratio": sharpe_ratio(strategy_return),
        "max_drawdown": max_drawdown(strategy_portfolio),
    }

## Baseline Models

The notebook includes:
- **Naive baseline**: tomorrow equals today
- **Faster ARIMA walk-forward**: refits every N steps instead of every single step

In [21]:
def naive_forecast(test_prev_close):
    return test_prev_close.copy()

def arima_walk_forward_fast(train_series, test_prev_close, order=(1, 1, 1), refit_every=ARIMA_REFIT_EVERY):
    history = list(train_series)
    preds = []
    model_fit = None

    for i, actual_prev_close in enumerate(test_prev_close):
        if i % refit_every == 0 or model_fit is None:
            try:
                model = ARIMA(
                    history,
                    order=order,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                model_fit = model.fit(method_kwargs={"warn_convergence": False})
            except Exception:
                model_fit = None

        try:
            forecast = model_fit.forecast()[0] if model_fit is not None else history[-1]
        except Exception:
            forecast = history[-1]

        preds.append(forecast)
        history.append(actual_prev_close)

    return np.array(preds)

## Sequence Builder

In [22]:
def create_sequences(X, y, sequence_length):
    X_seq, y_seq = [], []
    for i in range(sequence_length, len(X)):
        X_seq.append(X[i-sequence_length:i])
        y_seq.append(y[i])
    return np.array(X_seq), np.array(y_seq)

## Attention-Based PyTorch LSTM

This version uses:
- multi-layer LSTM
- attention mechanism over timesteps
- fully connected output layer
- support for both regression and classification targets

In [23]:
class LSTMWithAttention(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, target_mode="price"):
        super().__init__()
        self.target_mode = target_mode

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            batch_first=True
        )

        self.norm      = nn.LayerNorm(hidden_size)
        self.attention = nn.Linear(hidden_size, 1)
        self.fc        = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        lstm_out     = self.norm(lstm_out)          # stabilise gradients
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out          = self.fc(context)
        return out

## LSTM Training with Scheduler and Early Stopping

This training routine includes:
- train/validation split
- learning rate scheduler
- early stopping
- support for regression or direction classification

In [24]:
def train_lstm_and_predict_pytorch(
    df,
    split_idx,
    ticker,
    target_mode="price",
    sequence_length=30,
    epochs=60,
    batch_size=64,
    lr=1e-3
):
    feature_cols = [
        "Open", "High", "Low", "Close", "Volume",
        "return_1d", "return_5d", "return_10d", "return_20d", "intraday_return",
        "close_lag_1", "close_lag_2", "close_lag_3", "close_lag_5", "close_lag_10",
        "return_lag_1", "return_lag_2", "return_lag_3", "return_lag_5", "return_lag_10",
        "ma_5", "ma_10", "ma_20", "ma_30", "ma_cross",
        "momentum_5", "momentum_10",
        "volatility_5", "volatility_10", "volatility_20",
        "hl_range", "oc_range",
        "volume_change", "volume_ma_5", "volume_ma_20",
        "price_to_ma20", "vol_ratio", "rsi_14", "rsi_centered", "macd", "macd_signal", "macd_hist",
        "bb_upper", "bb_lower", "bb_width", "day_of_week", "month"
    ]

    if ticker in ("GLD", "SPY", "QQQ"):
        feature_cols.extend([
            "tlt_return_1d", "tlt_return_5d",
            "uup_return_1d", "uup_return_5d",
        ])

    X_raw = df[feature_cols].copy()
    y_raw = df[["target"]].copy()

    X_train_raw = X_raw.iloc[:split_idx]
    X_test_raw = X_raw.iloc[split_idx:]
    y_train_raw = y_raw.iloc[:split_idx]
    y_test_raw = y_raw.iloc[split_idx:]

    if len(X_train_raw) == 0 or len(X_test_raw) == 0:
        raise ValueError("Empty train/test split after feature engineering.")

    x_scaler = MinMaxScaler()
    X_train_scaled = x_scaler.fit_transform(X_train_raw)
    X_test_scaled = x_scaler.transform(X_test_raw)

    if target_mode == "price":
        y_scaler = MinMaxScaler()
        y_train_scaled = y_scaler.fit_transform(y_train_raw)
        y_test_scaled = y_scaler.transform(y_test_raw)
    else:
        y_scaler = None
        y_train_scaled = y_train_raw.values.astype(np.float32)
        y_test_scaled = y_test_raw.values.astype(np.float32)

    X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, sequence_length)
    X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, sequence_length)

    if len(X_train_seq) < 10 or len(X_test_seq) < 10:
        raise ValueError("Not enough sequences for LSTM training.")

    val_size = max(1, int(0.1 * len(X_train_seq)))
    X_val_seq = X_train_seq[-val_size:]
    y_val_seq = y_train_seq[-val_size:]
    X_train_seq2 = X_train_seq[:-val_size]
    y_train_seq2 = y_train_seq[:-val_size]

    X_train_t = torch.tensor(X_train_seq2, dtype=torch.float32)
    X_val_t = torch.tensor(X_val_seq, dtype=torch.float32)
    X_test_t = torch.tensor(X_test_seq, dtype=torch.float32)

    if target_mode == "price":
        y_train_t = torch.tensor(y_train_seq2, dtype=torch.float32)
    else:
        y_train_smooth = y_train_seq2 * 0.8 + 0.1
        y_train_t = torch.tensor(y_train_smooth, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_seq, dtype=torch.float32)

    if len(X_train_seq2) > 0:
        sample_weights = np.exp(np.linspace(-1, 0, len(X_train_seq2))).astype(np.float64)
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
    else:
        sampler = None

    train_loader = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size,
        sampler=sampler,
        shuffle=False if sampler is not None else False
    )

    model = LSTMWithAttention(
        input_size=X_train_t.shape[2],
        hidden_size=64,
        num_layers=2,
        dropout=0.2,
        target_mode=target_mode
    ).to(DEVICE)

    if target_mode == "price":
        criterion = nn.MSELoss()
    else:
        # Plain BCEWithLogitsLoss — markets are ~50/50 up/down so pos_weight
        # is not needed and overcorrection causes model collapse on some tickers.
        criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        patience=5,
        factor=0.5
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t.to(DEVICE))
            val_loss = criterion(val_logits, y_val_t.to(DEVICE)).item()

        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= EARLY_STOP_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t.to(DEVICE)).cpu().numpy().flatten()

    if target_mode == "price":
        pred = y_scaler.inverse_transform(test_logits.reshape(-1, 1)).flatten()
        actual = y_scaler.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()
        return pred, actual, None, None

    # Threshold is tuned on the validation slice carved from training data.
    # This is acceptable for quick research, but it can still overfit and may not generalize.
    val_probs = torch.sigmoid(model(X_val_t.to(DEVICE))).detach().cpu().numpy().flatten()
    val_actual = y_val_seq.flatten().astype(int)

    threshold_grid = np.arange(0.30, 0.71, 0.02)
    best_threshold = 0.5
    best_threshold_score = -1.0
    for threshold in threshold_grid:
        candidate_pred = (val_probs >= threshold).astype(int)
        score = accuracy_score(val_actual, candidate_pred)
        if score > best_threshold_score:
            best_threshold_score = score
            best_threshold = float(threshold)

    pred_probs = torch.sigmoid(torch.tensor(test_logits)).numpy()
    pred = (pred_probs >= best_threshold).astype(int)
    actual = y_test_seq.flatten().astype(int)

    return pred, actual, pred_probs, best_threshold

## End-to-End Evaluation for One Ticker

This function:
- builds features
- runs naive baseline
- runs faster ARIMA
- runs attention-based PyTorch LSTM
- computes metrics
- returns a comparison table

In [25]:
def evaluate_single_split(df, ticker, split_idx):
    train_close = df["Close"].iloc[:split_idx].values
    test_close = df["Close"].iloc[split_idx:].values
    test_next_close = df["next_close"].iloc[split_idx:].values
    test_return_1d = df["return_1d"].iloc[split_idx:].values

    prev_close = test_close.copy()
    naive_pred = naive_forecast(prev_close)
    arima_pred = arima_walk_forward_fast(
        train_close,
        prev_close,
        order=ARIMA_ORDER,
        refit_every=ARIMA_REFIT_EVERY
    )
    lstm_pred_full, lstm_actual_full, lstm_probs, lstm_threshold = train_lstm_and_predict_pytorch(
        df,
        split_idx,
        ticker=ticker,
        target_mode=TARGET_MODE,
        sequence_length=SEQUENCE_LENGTH,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LEARNING_RATE
    )

    eval_len = len(lstm_actual_full)
    prev_close_eval = prev_close[SEQUENCE_LENGTH:SEQUENCE_LENGTH + eval_len]
    actual_next_eval = test_next_close[SEQUENCE_LENGTH:SEQUENCE_LENGTH + eval_len]
    naive_pred_eval = naive_pred[SEQUENCE_LENGTH:SEQUENCE_LENGTH + eval_len]
    arima_pred_eval = arima_pred[SEQUENCE_LENGTH:SEQUENCE_LENGTH + eval_len]
    prev_return_eval = test_return_1d[SEQUENCE_LENGTH:SEQUENCE_LENGTH + eval_len]

    rows = []

    if TARGET_MODE == "price":
        actual_eval = lstm_actual_full[:eval_len]
        models = {
            "Naive": naive_pred_eval,
            "ARIMA": arima_pred_eval,
            "LSTM_Attention_PyTorch": lstm_pred_full[:eval_len],
        }
        for model_name, pred in models.items():
            rmse = math.sqrt(mean_squared_error(actual_eval, pred))
            mae = mean_absolute_error(actual_eval, pred)
            mape_value = mape(actual_eval, pred)
            dir_acc = directional_accuracy_from_prev(prev_close_eval, actual_eval, pred)
            strat = calculate_strategy_metrics(prev_close_eval, actual_eval, pred, INITIAL_CAPITAL)
            rows.append({
                "Ticker": ticker,
                "Model": model_name,
                "RMSE": rmse,
                "MAE": mae,
                "MAPE_%": mape_value,
                "Directional_Accuracy_%": dir_acc,
                "Strategy_Return_%": strat["strategy_cum_return_pct"],
                "BuyHold_Return_%": strat["buy_hold_cum_return_pct"],
                "Sharpe_Ratio": strat["sharpe_ratio"],
                "Max_Drawdown_%": strat["max_drawdown"] * 100,
                "Coverage_%": 100.0,
            })
        return pd.DataFrame(rows)

    actual_eval = lstm_actual_full[:eval_len]
    naive_direction = (prev_return_eval > 0).astype(int)
    arima_direction = (arima_pred_eval > prev_close_eval).astype(int)

    models = {
        "Naive": naive_direction,
        "ARIMA": arima_direction,
        "LSTM_Attention_PyTorch": lstm_pred_full[:eval_len],
    }

    for model_name, pred in models.items():
        dir_acc = accuracy_score(actual_eval, pred) * 100
        strat = calculate_strategy_metrics_from_direction(
            prev_close_eval,
            actual_next_eval,
            pred,
            INITIAL_CAPITAL
        )
        row = {
            "Ticker": ticker,
            "Model": model_name,
            "RMSE": np.nan,
            "MAE": np.nan,
            "MAPE_%": np.nan,
            "Directional_Accuracy_%": dir_acc,
            "Strategy_Return_%": strat["strategy_cum_return_pct"],
            "BuyHold_Return_%": strat["buy_hold_cum_return_pct"],
            "Sharpe_Ratio": strat["sharpe_ratio"],
            "Max_Drawdown_%": strat["max_drawdown"] * 100,
            "Coverage_%": 100.0,
        }
        if model_name == "LSTM_Attention_PyTorch" and lstm_threshold is not None:
            row["Decision_Threshold"] = float(lstm_threshold)
        rows.append(row)

    if lstm_probs is not None:
        high_conf_mask = (lstm_probs >= CONFIDENCE_THRESHOLD) | (lstm_probs <= (1 - CONFIDENCE_THRESHOLD))
        if high_conf_mask.sum() > 10:
            hc_pred = lstm_pred_full[:eval_len][high_conf_mask]
            hc_actual = actual_eval[high_conf_mask]
            hc_prev_close = prev_close_eval[high_conf_mask]
            hc_actual_next = actual_next_eval[high_conf_mask]
            hc_acc = accuracy_score(hc_actual, hc_pred) * 100
            hc_coverage = high_conf_mask.mean() * 100
            hc_strat = calculate_strategy_metrics_from_direction(
                hc_prev_close,
                hc_actual_next,
                hc_pred,
                INITIAL_CAPITAL
            )
            rows.append({
                "Ticker": ticker,
                "Model": f"LSTM_HighConf_{CONFIDENCE_THRESHOLD}",
                "RMSE": np.nan,
                "MAE": np.nan,
                "MAPE_%": np.nan,
                "Directional_Accuracy_%": hc_acc,
                "Strategy_Return_%": hc_strat["strategy_cum_return_pct"],
                "BuyHold_Return_%": hc_strat["buy_hold_cum_return_pct"],
                "Sharpe_Ratio": hc_strat["sharpe_ratio"],
                "Max_Drawdown_%": hc_strat["max_drawdown"] * 100,
                "Coverage_%": hc_coverage,
                "Decision_Threshold": float(lstm_threshold) if lstm_threshold is not None else np.nan,
            })

        consensus_mask = np.abs(lstm_probs - 0.5) >= CONSENSUS_GAP
        consensus_mask &= (lstm_pred_full[:eval_len] == arima_direction)
        if consensus_mask.sum() > 10:
            consensus_pred = lstm_pred_full[:eval_len][consensus_mask]
            consensus_actual = actual_eval[consensus_mask]
            consensus_prev_close = prev_close_eval[consensus_mask]
            consensus_actual_next = actual_next_eval[consensus_mask]
            consensus_acc = accuracy_score(consensus_actual, consensus_pred) * 100
            consensus_coverage = consensus_mask.mean() * 100
            consensus_strat = calculate_strategy_metrics_from_direction(
                consensus_prev_close,
                consensus_actual_next,
                consensus_pred,
                INITIAL_CAPITAL
            )
            rows.append({
                "Ticker": ticker,
                "Model": "LSTM_ARIMA_Consensus",
                "RMSE": np.nan,
                "MAE": np.nan,
                "MAPE_%": np.nan,
                "Directional_Accuracy_%": consensus_acc,
                "Strategy_Return_%": consensus_strat["strategy_cum_return_pct"],
                "BuyHold_Return_%": consensus_strat["buy_hold_cum_return_pct"],
                "Sharpe_Ratio": consensus_strat["sharpe_ratio"],
                "Max_Drawdown_%": consensus_strat["max_drawdown"] * 100,
                "Coverage_%": consensus_coverage,
                "Decision_Threshold": float(lstm_threshold) if lstm_threshold is not None else np.nan,
            })

    return pd.DataFrame(rows)


def aggregate_walk_forward_results(results):
    metric_cols = [
        "RMSE", "MAE", "MAPE_%", "Directional_Accuracy_%",
        "Strategy_Return_%", "BuyHold_Return_%", "Sharpe_Ratio",
        "Max_Drawdown_%", "Coverage_%", "Decision_Threshold"
    ]
    available_metrics = [col for col in metric_cols if col in results.columns]
    aggregated = (
        results.groupby(["Ticker", "Model"], as_index=False)[available_metrics]
        .mean(numeric_only=True)
    )
    fold_counts = results.groupby(["Ticker", "Model"]).size().reset_index(name="Fold_Count")
    aggregated = aggregated.merge(fold_counts, on=["Ticker", "Model"], how="left")

    for col in available_metrics:
        if col in aggregated.columns:
            aggregated[col] = aggregated[col].round(3)
    return aggregated


def evaluate_ticker_robust(market_df, ticker):
    df = build_features_for_ticker(market_df, ticker, target_mode=TARGET_MODE)
    print(f"[{ticker}] feature engineering complete | rows={len(df)}", flush=True)

    max_start = len(df) - MIN_TEST_SIZE
    split_candidates = np.linspace(MIN_TRAIN_SIZE, max_start, WALK_FORWARD_SPLITS, dtype=int)
    split_candidates = sorted(set(int(x) for x in split_candidates if MIN_TRAIN_SIZE <= x < len(df) - MIN_TEST_SIZE + 1))

    if not split_candidates:
        fallback_split = int(len(df) * TRAIN_RATIO)
        split_candidates = [fallback_split]

    all_fold_results = []
    for fold_number, split_idx in enumerate(split_candidates, start=1):
        print(f"[{ticker}] walk-forward fold {fold_number}/{len(split_candidates)} | split={split_idx}", flush=True)
        fold_result = evaluate_single_split(df, ticker, split_idx)
        fold_result["Fold"] = fold_number
        all_fold_results.append(fold_result)

    combined = pd.concat(all_fold_results, ignore_index=True)
    summary = aggregate_walk_forward_results(combined)
    print(f"[{ticker}] finished", flush=True)
    return summary

## Parallel Execution Across Tickers

On some notebook environments, process-based parallelism can be flaky. The helper below wraps the execution in a function, which is usually more stable on Windows + Jupyter. If this still misbehaves, run the sequential fallback cell instead.

In [26]:
from concurrent.futures import ProcessPoolExecutor, as_completed

def evaluate_wrapper(args):
    ticker, market_df = args
    try:
        return evaluate_ticker_robust(market_df, ticker)
    except Exception as e:
        print(f"Error for {ticker}: {e}", flush=True)
        return None

def run_parallel(market_data, tickers):
    """
    Parallel ticker evaluation.

    Preferred when running in Colab or as a .py script.
    If multiprocessing behaves oddly in a notebook environment,
    use the sequential fallback instead.
    """
    all_results = []

    with ProcessPoolExecutor(max_workers=min(4, len(tickers))) as executor:
        futures = [
            executor.submit(evaluate_wrapper, (ticker, market_data))
            for ticker in tickers
        ]

        for future in as_completed(futures):
            result = future.result()
            if result is not None and not result.empty:
                all_results.append(result)

    if not all_results:
        raise ValueError("No results were returned from parallel execution.")

    return pd.concat(all_results, ignore_index=True)

# Run parallel (preferred for Colab / .py scripts):
final_results = run_parallel(market_data, TICKERS)
final_results

[GLD] feature engineering complete | rows=2731
[GLD] walk-forward fold 1/3 | split=900
[SPY] feature engineering complete | rows=2731
[SPY] walk-forward fold 1/3 | split=900
[QQQ] feature engineering complete | rows=2731
[QQQ] walk-forward fold 1/3 | split=900[AAPL] feature engineering complete | rows=2731

[AAPL] walk-forward fold 1/3 | split=900
[SPY] walk-forward fold 2/3 | split=1725
[QQQ] walk-forward fold 2/3 | split=1725
[GLD] walk-forward fold 2/3 | split=1725
[AAPL] walk-forward fold 2/3 | split=1725
[SPY] walk-forward fold 3/3 | split=2551
[QQQ] walk-forward fold 3/3 | split=2551
[SPY] finished
[GLD] walk-forward fold 3/3 | split=2551
[QQQ] finished
[AAPL] walk-forward fold 3/3 | split=2551
[GLD] finished
[AAPL] finished


,Ticker,Model,RMSE,MAE,MAPE_%,Directional_Accuracy_%,Strategy_Return_%,BuyHold_Return_%,Sharpe_Ratio,Max_Drawdown_%,Coverage_%,Decision_Threshold,Fold_Count
0,SPY,ARIMA,NaN,NaN,NaN,51.801,606.367,6931.533,3.648,-40.984,100.000,NaN,3
1,SPY,LSTM_ARIMA_Consensus,NaN,NaN,NaN,69.360,445.650,434.718,6.506,-36.893,36.280,0.393,3
2,SPY,LSTM_Attention_PyTorch,NaN,NaN,NaN,62.089,7036.557,6931.533,3.162,-56.072,100.000,0.393,3
3,SPY,LSTM_HighConf_0.6,NaN,NaN,NaN,61.818,3280.567,3231.162,2.239,-65.881,80.788,0.300,2
4,SPY,Naive,NaN,NaN,NaN,49.316,267.892,6931.533,1.757,-44.194,100.000,NaN,3
5,QQQ,ARIMA,NaN,NaN,NaN,49.265,2038.042,29998.479,3.140,-41.749,100.000,NaN,3
6,QQQ,LSTM_ARIMA_Consensus,NaN,NaN,NaN,50.888,666.242,815.308,1.830,-34.114,34.435,0.453,3
7,QQQ,LSTM_Attention_PyTorch,NaN,NaN,NaN,49.717,8700.740,29998.479,2.263,-51.833,100.000,0.453,3
8,QQQ,LSTM_HighConf_0.6,NaN,NaN,NaN,58.473,8155.178,8155.178,1.496,-71.531,35.640,0.440,2
9,QQQ,Naive,NaN,NaN,NaN,50.763,611.157,29998.479,2.037,-53.395,100.000,NaN,3


## Sequential Fallback (Safer in Jupyter)

Use this if parallel execution hangs in your notebook environment.

In [ ]:
all_results = []

for ticker in TICKERS:
    try:
        print(f"\n===== Running {ticker} =====", flush=True)
        result = evaluate_ticker_robust(market_data, ticker)
        if result is not None:
            all_results.append(result)
    except Exception as e:
        print(f"Error for {ticker}: {e}")

final_results = pd.concat(all_results, ignore_index=True)
final_results


===== Running GLD =====
[GLD] feature engineering complete | rows=2731
[GLD] walk-forward fold 1/3 | split=900


## Save Final Comparison Table

In [ ]:
final_results.to_csv("quant_forecasting_model_comparison_robust.csv", index=False)
print("Saved results to quant_forecasting_model_comparison_robust.csv")
final_results

## Conclusion

This upgraded notebook moves beyond a basic forecasting notebook by adding:

- 10 years of market history
- richer financial indicators
- stronger LSTM architecture
- faster ARIMA benchmarking
- optional classification target
- caching for reproducibility and speed

It is designed to resemble a compact **quant research workflow** rather than a simple classroom project.

## Reproducibility

- Data is pulled programmatically from `yfinance`
- Downloads are cached locally
- Random seeds are fixed
- The notebook runs end-to-end without external CSV files